# 투수 중심 MLB Statcast EDA

각 투수를 중심으로 리그 내 위치, 구종 구성, 카운트·타자 좌우 스플릿, 구속·회전·무브먼트, 릴리스 포인트, 월별 추세를 분석합니다. 이 노트북은 기술통계용이므로 현재 투구의 물리값과 결과를 함께 사용하며, 모델 학습용 누수 방지 데이터와 목적이 다릅니다.

In [ ]:
# 최초 1회만 실행
%pip install -q pandas numpy pyarrow matplotlib seaborn

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from build_statcast_strike_dataset import BuildConfig, RAW_COLUMNS, audit_raw, load_raw
from pitcher_eda import (
    count_profile, handedness_profile, monthly_profile, pitch_type_profile,
    pitcher_overview, prepare_pitcher_eda, select_pitcher,
)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 100)

## 1. 데이터 로드와 투수 선택

`PITCHER_ID=None`이면 표본이 가장 많은 투수를 자동 선택합니다. 특정 투수는 Statcast의 MLBAM ID를 입력하세요. `MIN_PITCHES`는 리그 비교에 포함할 최소 투구 수입니다.

In [ ]:
PITCHER_ID = None
MIN_PITCHES = 300
RAW_DIR = Path('data/statcast_raw_sequence')

config = BuildConfig(raw_dir=RAW_DIR)
raw = load_raw(config, columns=RAW_COLUMNS)
display(audit_raw(raw))
eda = prepare_pitcher_eda(raw)
league = pitcher_overview(eda, min_pitches=MIN_PITCHES)
pitcher_id, pitcher_name, pitcher_df = select_pitcher(eda, PITCHER_ID)
print(f'선택 투수: {pitcher_name} ({pitcher_id}) | {len(pitcher_df):,}구 | {pitcher_df.game_pk.nunique():,}경기')

## 2. 리그 내 위치

투구 수와 스트라이크율을 함께 보면 표본 규모와 공격성을 동시에 비교할 수 있습니다. 점 크기는 헛스윙률입니다.

In [ ]:
display(league.head(20).style.format({
    'strike_rate': '{:.1%}', 'whiff_rate': '{:.1%}',
    'called_strike_rate': '{:.1%}', 'avg_velocity': '{:.1f}', 'avg_spin': '{:.0f}',
}))

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=league, x='pitches', y='strike_rate', size='whiff_rate',
                sizes=(25, 220), alpha=.65, legend='brief', ax=ax)
selected_row = league[league['pitcher'].eq(pitcher_id)]
if not selected_row.empty:
    x, y = selected_row.iloc[0][['pitches', 'strike_rate']]
    ax.scatter([x], [y], s=220, marker='*', color='crimson', zorder=5)
    ax.annotate(pitcher_name, (x, y), xytext=(8, 8), textcoords='offset points')
ax.set(xscale='log', xlabel='Pitches (log scale)', ylabel='Strike rate', title='Pitcher workload and strike rate')
ax.yaxis.set_major_formatter(lambda value, _: f'{value:.0%}')
plt.show()

## 3. 구종 구성과 구종별 성과

구종 사용률은 투수의 기본 전략을, 스트라이크율·헛스윙률·구속·무브먼트는 각 구종의 성격을 보여줍니다.

In [ ]:
pitch_profile = pitch_type_profile(pitcher_df, min_pitches=max(5, int(len(pitcher_df) * .005)))
display(pitch_profile.style.format({
    'usage_rate': '{:.1%}', 'strike_rate': '{:.1%}', 'whiff_rate': '{:.1%}',
    'avg_velocity': '{:.1f}', 'avg_spin': '{:.0f}', 'pfx_x_in': '{:.1f}', 'pfx_z_in': '{:.1f}',
    'release_x': '{:.2f}', 'release_z': '{:.2f}',
}))

fig, ax = plt.subplots(figsize=(9, max(4, .55 * len(pitch_profile))))
sns.barplot(data=pitch_profile, y='pitch_type', x='usage_rate', order=pitch_profile['pitch_type'], ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', labels=[f'{v:.1%}' for v in pitch_profile['usage_rate']], padding=3)
ax.set(xlabel='Usage rate', ylabel='Pitch type', title=f'{pitcher_name}: pitch mix')
ax.xaxis.set_major_formatter(lambda value, _: f'{value:.0%}')
plt.show()

In [ ]:
sample = pitcher_df.dropna(subset=['pfx_x_in', 'pfx_z_in', 'pitch_type']).sample(
    n=min(5000, pitcher_df[['pfx_x_in', 'pfx_z_in', 'pitch_type']].dropna().shape[0]), random_state=42
)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=sample, x='pfx_x_in', y='pfx_z_in', hue='pitch_type', alpha=.35, s=22, ax=axes[0])
axes[0].axhline(0, color='grey', lw=.8); axes[0].axvline(0, color='grey', lw=.8)
axes[0].set(title='Movement profile', xlabel='Horizontal break (in)', ylabel='Induced vertical break (in)')
release = pitcher_df.dropna(subset=['release_pos_x', 'release_pos_z', 'pitch_type'])
release = release.sample(n=min(5000, len(release)), random_state=42)
sns.scatterplot(data=release, x='release_pos_x', y='release_pos_z', hue='pitch_type', alpha=.3, s=22, ax=axes[1], legend=False)
axes[1].set(title='Release-point consistency', xlabel='Release X (ft)', ylabel='Release Z (ft)')
plt.tight_layout(); plt.show()

## 4. 카운트와 타자 좌우 스플릿

카운트별 표본 수를 스트라이크율과 함께 확인해야 작은 표본에서 생기는 극단값을 피할 수 있습니다.

In [ ]:
counts = count_profile(pitcher_df)
rate_matrix = counts.pivot(index='strikes', columns='balls', values='strike_rate').sort_index(ascending=False)
n_matrix = counts.pivot(index='strikes', columns='balls', values='pitches').sort_index(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(rate_matrix, annot=True, fmt='.1%', vmin=0, vmax=1, cmap='viridis', ax=axes[0])
sns.heatmap(n_matrix, annot=True, fmt='.0f', cmap='Blues', ax=axes[1])
axes[0].set_title('Strike rate by count'); axes[1].set_title('Pitch count by count')
plt.tight_layout(); plt.show()

hand = handedness_profile(pitcher_df)
display(hand.style.format({'strike_rate': '{:.1%}', 'whiff_rate': '{:.1%}', 'avg_velocity': '{:.1f}'}))
long = hand.melt(id_vars='stand', value_vars=['strike_rate', 'whiff_rate'], var_name='metric', value_name='rate')
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=long, x='stand', y='rate', hue='metric', ax=ax)
ax.set(xlabel='Batter side', ylabel='Rate', title='Platoon split')
ax.yaxis.set_major_formatter(lambda value, _: f'{value:.0%}')
plt.show()

## 5. 월별 추세와 로케이션

월별 구속·스트라이크율 변화는 시즌 중 피로, 역할 변화 또는 구종 조정의 후보 신호입니다. 로케이션 차트는 콜드 피치에서 위치별 스트라이크 판정 비율을 나타냅니다.

In [ ]:
monthly = monthly_profile(pitcher_df)
display(monthly.style.format({'strike_rate': '{:.1%}', 'whiff_rate': '{:.1%}', 'avg_velocity': '{:.1f}', 'avg_spin': '{:.0f}'}))
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
sns.lineplot(data=monthly, x='month', y='avg_velocity', marker='o', ax=axes[0])
sns.lineplot(data=monthly, x='month', y='strike_rate', marker='o', ax=axes[1])
axes[0].set(ylabel='Average velocity (mph)', title=f'{pitcher_name}: monthly trend')
axes[1].set(ylabel='Strike rate', xlabel='Month')
axes[1].yaxis.set_major_formatter(lambda value, _: f'{value:.0%}')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
called = pitcher_df[pitcher_df['is_called_pitch'].eq(1)].dropna(subset=['plate_x', 'plate_z', 'is_called_strike'])
fig, ax = plt.subplots(figsize=(7, 6))
hb = ax.hexbin(called['plate_x'], called['plate_z'], C=called['is_called_strike'],
               reduce_C_function=np.mean, gridsize=22, mincnt=5, vmin=0, vmax=1, cmap='viridis')
fig.colorbar(hb, ax=ax, label='Called-strike rate')
ax.set(xlim=(-2.2, 2.2), ylim=(0.5, 4.5), xlabel='Plate X (ft)', ylabel='Plate Z (ft)',
       title='Called-strike rate by location (minimum 5 pitches per hex)')
plt.show()

## 6. 요약표 저장

선택 투수의 표본을 재현할 수 있도록 투수 ID가 포함된 폴더에 집계표를 저장합니다.

In [ ]:
root_report_dir = Path('reports/pitcher_eda')
report_dir = root_report_dir / str(pitcher_id)
report_dir.mkdir(parents=True, exist_ok=True)
league.to_csv(root_report_dir / 'league_pitcher_overview.csv', index=False, encoding='utf-8-sig')
pitch_profile.to_csv(report_dir / 'pitch_type_profile.csv', index=False, encoding='utf-8-sig')
counts.to_csv(report_dir / 'count_profile.csv', index=False, encoding='utf-8-sig')
hand.to_csv(report_dir / 'handedness_profile.csv', index=False, encoding='utf-8-sig')
monthly.to_csv(report_dir / 'monthly_profile.csv', index=False, encoding='utf-8-sig')
print('저장 위치:', report_dir.resolve())